# Baqaee & Farhi (2022) — Calibration Grid (Phase 5)

This notebook runs the full calibration grid: the four nested loops of
`Master_file_3.m` (loop × elasticity × shock_type × htm_share × t_grid).

**Memory note:** Each solve is a 668-variable FD Jacobian system. The full
grid (~230 solves for loop=1, ~96 for loop=2) fits on the host Mac (32 GB)
but may OOM a 5 GB container. Run this notebook on the Mac.

**Output:** CSVs are written to `data/results/` — one folder per cell
(loop/el/st/s/timeseries.csv + sector_prices.csv), plus summary files
(`summary_loop1.csv`, `summary_loop2_htm.csv`, `baseline_fit.csv`).
These CSVs can be loaded in the container for analysis.


In [ ]:
# Setup
cd(dirname(Base.active_project()))
using Printf, LinearAlgebra, Statistics
include("src/io_table.jl")
include("src/shocks.jl")
include("src/network.jl")
include("src/model.jl")
include("src/calibration_grid.jl")

println("Modules loaded ✅")


---
## 1. Single-Cell Test

Run one cell (loop=1, elasticity=1, shock_type=1, htm_share=0) to verify
the driver works before scaling up.


In [ ]:
io = load_io_table(joinpath("data", "IO_data_2018.mat"); N=66, year=2015)
shocks = load_shocks("data"; N=66)
sf = build_standard_form(io)

t_grid = [0.0, 0.01, 0.05, 0.10, 0.25, 0.5, 0.75, 1.0]
res, _ = solve_cell(io, shocks, sf; elasticity=1, shock_type=1,
                    htm_share=0.0, t_grid=t_grid)
dR = delta_r_gdp(res)
@printf("Single cell: RGDP=%.4f  \u0394RGDP=%.2f%%  nom=%.4f  retcodes=%s\n", res["GDP"][end], -100*dR[end], res["nominal_GDP"][end], res["retcodes"])


---
## 2. Full Grid (Loop 1)

Run the full loop=1 grid: elasticity=1:2 × shock_type=1:5 × s=1.
This reproduces Figures 2–3 and Tables A1–A2.

> ⚠️ This takes ~5–15 minutes on a 32 GB Mac. Each cell writes its own
> CSV immediately, so partial results survive if interrupted.


In [ ]:
outdir = "data/results"
summary = run_calibration_grid(outdir; loops=[1], verbose=true)

println("\n=== Summary (loop=1) ===\n")
println("shock_type | RGDP_bench | Infl_bench | Unemp_bench | RGDP_CD | Infl_CD | Unemp_CD")
for st in 1:5
    @printf("  %-12s | %8.1f%% | %8.1f%% | %9.1f%% | %7.1f%% | %7.1f%% | %8.1f%%\n",
            ["baseline", "supply", "demand", "agg_demand", "supply+sec"][st],
            -summary["RGDP_graph"][1, st],
            summary["Inflation_graph"][1, st],
            summary["Unemp_graph"][1, st],
            -summary["RGDP_graph"][2, st],
            summary["Inflation_graph"][2, st],
            summary["Unemp_graph"][2, st])
end
println(")\nCSVs written to ", joinpath(pwd(), outdir))


---
## 3. HtM Sweep (Loop 2)

Run loop=2 (elasticity=1:2, shock_type=1, htm_share=0:0.2:1). This
reproduces Figure 4. Also writes summary CSVs.


In [ ]:
htm_summary = run_calibration_grid(outdir; loops=[2], verbose=true)

println("\n=== HtM sweep (loop=2) ===\n")
println("htm_share | RGDP_bench | Infl_bench | Unemp_bench | RGDP_CD | Infl_CD | Unemp_CD")
for (i, sh) in enumerate(0.0:0.2:1.0)
    @printf("  %8.1f | %8.2f%% | %8.2f%% | %9.2f%% | %7.2f%% | %7.2f%% | %8.2f%%\n",
            sh,
            -100*htm_summary["RGDP_graph_htm"][1, i],
            100*htm_summary["Inflation_graph_htm"][1, i],
            100*htm_summary["Unemp_graph_htm"][1, i],
            -100*htm_summary["RGDP_graph_htm"][2, i],
            100*htm_summary["Inflation_graph_htm"][2, i],
            100*htm_summary["Unemp_graph_htm"][2, i])
end


---
## 4. Baseline Fit (Parts A, B, D)

The baseline fit CSVs are already written by `run_calibration_grid`.
`baseline_fit.csv` contains sector-level data for the Part-B out-of-sample
fit (PPI, wage, hours), the Part-D tightness/slackness decomposition, and
the Part-A calibration targets.

Use this file in a separate analysis notebook or plotting script.


In [ ]:
println("✅ Phase 5 complete. CSVs in ", joinpath(pwd(), outdir))
